# Screen sample repetitions

Finds samples that are in more than one Mad4HaTter allele table (sequenced at different moments) and reports their depth in each table. Only samples are screened; controls are left unchanged. It can also write one allele table, all input tables combined, in which those samples are merged, cut to their best table, or renamed. This is the notebook version of `screen_sample_repetitions.py`, and both give the same output files.

**How to use:** edit the **Settings** cell, then run all cells (**Run All**). Relative paths are read from the folder of this notebook.

## Prerequisites

- Python 3.9 or newer
- pandas (`pip install pandas` or `conda install pandas`)
- Jupyter: JupyterLab, Jupyter Notebook or VS Code

## Settings

**Allele tables (`ALLELE_FILES`):** one or more `.txt`, `.tsv` or `.csv` files (optionally `.gz`), or folders. In a folder, every file whose name starts with `allele` is read (for example `allele_table_run1.txt` or `allele_data.tsv.gz`). All the tables are combined into one. Column names are matched ignoring case.

- **Required columns:** the sample ID (`SampleID`, `sample_id`, `sample_name` or `sample`), `locus` and `reads`.
- **Optional columns:** `asv`, `NIDA`, `norm.reads.locus` and `n.alleles`.
- **Origin of each sample:** the allele table it comes from. A sample found in more than one table is a repeat, so give two or more tables (or a folder with them).
- **What gets screened:** only samples, that is IDs that standardise to a number: `N2076817-1_S3_L001`, `N2076817_1_S12`, `N2076817-1` and `2076817.1` are all sample `2076817.1`. Other IDs are controls (`N3D7-10K`, `NDd2_100k`, ...): they are not screened and are left unchanged.
- **`Allele_Table`:** added to both outputs, with the file name of the table each row comes from (its path if two tables have the same file name).
- **Outputs:** write them outside the folder of the allele tables, or a later run will read an old output as one more table.

**Options:** set at most one of `MERGE`, `BEST_COVERAGE` and `KEEP_DUPS` to `True`.

- **Best table:** the one with the most loci above `MIN_DEPTH` reads. Ties go to the table with more reads.
- **`KEEP_DUPS`:** the letter goes after `_S*_L001` in the sample ID and at the end of `NIDA`, for example `N1910666-3_S59_L001_a` and `1910666.3_a`.
- **`MERGE`:** needs an `asv` column when the allele table is written, because allele names are numbered separately in each table.

In [ ]:
# ---- Input files -----------------------------------------------------------
ALLELE_FILES = "all_runs/"           # one or more tables or folders, e.g. ["allele_run1.txt", "allele_run2.txt"] or "all_runs/"
ALLELES_ID_COL = None                # sample ID column, only if not SampleID, sample_id, sample_name or sample

# ---- Output files ----------------------------------------------------------
OUT_FILE = "screening.tsv"           # screening table (.csv gives commas, anything else tabs)
OUT_ALLELES_FILE = None              # allele table with the option below applied, e.g. "allele_table_best_run.txt"; None = not written

# ---- Samples in more than one allele table: set at most one to True ---------
MERGE = False           # combine each sample's tables into one, adding up the reads
BEST_COVERAGE = False   # keep only the best table of each sample
KEEP_DUPS = False       # keep every table and add _a, _b, ... to the IDs (a = best table)

# ---- Screening -------------------------------------------------------------
MIN_DEPTH = 100         # reads a locus must exceed to count in Depth_Read_Loci
SHOW_ALL = False        # True: also list samples in only one allele table in the screening table

## Functions

These are the same functions as in `screen_sample_repetitions.py`.

In [ ]:
import gzip
import os
import re

import pandas as pd

# Column names are matched ignoring case (SampleID = sampleID, Reads = reads)
ID_NAMES = ["sampleid", "sample_id", "sample_name", "sample"]
# Standardised IDs of samples (2076817, 2076817.1); any other ID is a control
SAMPLE_ID_PATTERN = r"\d+(\.\d+)?"
# Files read from a folder in ALLELE_FILES
ALLELE_FILE_PATTERN = re.compile(r"^[Aa]llele.*\.(txt|tsv|csv)(\.gz)?$")
TABLE_COL = "Allele_Table"   # column naming the allele table of each row


def letter(rank):
    """0 -> a, 1 -> b, ..., 25 -> z, 26 -> aa."""
    s, n = "", rank + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(97 + r) + s
    return s


def add_letter(raw, tag):
    """Put tag right after _S12 / _S3_L001 if the ID has it, otherwise at the end:
    N2076817_1_S64__HFS23_NextSEQ01_271025 -> N2076817_1_S64_a__HFS23_NextSEQ01_271025."""
    m = re.search(r"_S\d+(?:_L\d+)?(?=__|$)", raw)
    return raw[:m.end()] + tag + raw[m.end():] if m else raw + tag


def number_text(v):
    return str(int(v)) if float(v).is_integer() else str(float(v))


def separator(path):
    return "," if path.lower().removesuffix(".gz").endswith(".csv") else "\t"


def shortlist(values, n=20):
    values = list(values)
    return ", ".join(map(str, values[:n])) + (" ..." if len(values) > n else "")


def join_unique(values):
    """'a', 'b;a', 'c', '' -> 'a;b;c' (first appearance order, empty values left out)."""
    return ";".join(dict.fromkeys(x for v in values for x in v.split(";") if x))


def join_sorted(values):
    return ";".join(sorted(set(values)))


def summarise(per_locus, keys, min_depth):
    """Reads, loci and deep loci for each group of per-locus read totals."""
    s = per_locus.groupby(keys).agg(
        Total_Reads=("reads", "sum"),
        Total_Loci=("reads", lambda v: int((v > 0).sum())),
        Depth_Read_Loci=("reads", lambda v: int((v > min_depth).sum()))).reset_index()
    s["Avg_Reads_per_Locus"] = (s.Total_Reads / s.Total_Loci.where(s.Total_Loci > 0)).round(2)
    return s


def normalise_id(raw):
    s = str(raw).strip()
    s = re.sub(r"__.*$", "", s)           # __MULE_NextSEQ03_300725
    s = re.sub(r"_S\d+(_L\d+)?$", "", s)  # _S12_L001
    m = re.match(r"^N?(\d+)(?:[-_.](\d+))?$", s)
    if m:
        base, dec = m.groups()
        return base if dec in (None, "0") else f"{base}.{dec}"
    return s.replace("-", "_")            # controls, e.g. N3D7-10K -> N3D7_10K


def read_table(path):
    if path.lower().endswith((".xlsx", ".xls")):
        raise ValueError(f"{path}: save it as .csv or tab-separated .txt first.")
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="latin-1") as fh:
        first = fh.readline()
    sep = "\t" if "\t" in first else ","
    return pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, encoding="latin-1")


def find_col(df, wanted, names, label):
    if wanted:
        if wanted not in df.columns:
            raise ValueError(f"Column '{wanted}' not found for {label}. Columns: {list(df.columns)}")
        return wanted
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n in lower:
            return lower[n]
    return None


def allele_files(paths):
    """Allele table files from the files and folders given. In a folder, every file whose name
    starts with "allele" and ends in .txt, .tsv or .csv (optionally .gz) is read."""
    files = {}
    for path in paths:
        if os.path.isdir(path):
            folder = path.rstrip("/") or path
            found = [os.path.join(folder, f) for f in sorted(os.listdir(folder))
                     if ALLELE_FILE_PATTERN.match(f) and not os.path.isdir(os.path.join(folder, f))]
            if not found:
                raise FileNotFoundError(f"{path}: no allele tables in this folder (files named allele*.txt, .tsv or "
                                        ".csv, optionally .gz).")
        elif os.path.isfile(path):
            found = [path]
        else:
            raise FileNotFoundError(f"Allele table or folder not found: {path}")
        for f in found:
            files.setdefault(os.path.abspath(f), f)
    return list(files.values())


def table_names(files):
    """File name of each allele table, or its path when two tables have the same file name."""
    base = [os.path.basename(f) for f in files]
    return [f if base.count(b) > 1 else b for f, b in zip(files, base)]


def read_alleles(files):
    """All allele tables as one, plus the name of its Allele_Table column.

    Column names that differ only in case (sampleID, SampleID) get the first table's spelling.
    Allele_Table holds the name of each row's table (kept where the tables already have one).
    """
    tables, spelling = [], {}
    for path in files:
        df = read_table(path)
        df = df.rename(columns={c: spelling[c.lower()] for c in df.columns
                                if c.lower() in spelling and spelling[c.lower()] not in df.columns})
        for c in df.columns:
            spelling.setdefault(c.lower(), c)
        tables.append(df)
    table_col = spelling.get(TABLE_COL.lower(), TABLE_COL)
    for df, name in zip(tables, table_names(files)):
        if table_col in df.columns:
            df[table_col] = df[table_col].where(df[table_col].str.strip() != "", name)
        else:
            df[table_col] = name
    # Columns missing from a table are left empty
    return pd.concat(tables, ignore_index=True).fillna(""), table_col


def resolve_alleles(al, c, links, repeated, ranks, mode):
    """Allele table with each sample in several tables merged, cut to its best table, or relabelled.

    c       column names in al: id, locus, reads, asv, run, nida, allele, norm, n_alleles, table
    links   row (index in al), SampleID and Allele_Table of every allele row of a sample
    repeated  rows left out when adding up reads (same sequence listed twice)
    ranks   SampleID, Allele_Table and Rank (0 = best) of the samples in several tables
    """
    dup = links.merge(ranks, on=["SampleID", "Allele_Table"])
    if dup.empty:   # no sample in more than one table
        return al
    if mode == "best_coverage":
        return al.drop(index=dup.row[dup.Rank > 0])

    if mode == "keep_dups":
        out = al.copy()
        tag = pd.Series(("_" + dup.Rank.map(letter)).to_numpy(), index=dup.row.to_numpy())
        for col in dict.fromkeys(x for x in (c["id"], c["nida"]) if x):
            vals = out.loc[tag.index, col]
            out.loc[tag.index, col] = [v if v == "" else add_letter(v, t) for v, t in zip(vals, tag)]
        return out

    # merge: one row per sample, locus and sequence; the other columns come
    # from the best table that has that sequence
    locus, reads = c["locus"], c["reads"]
    keys = ["_sid", locus, c["asv"]]
    use = dup[~dup.row.isin(repeated)]
    part = al.loc[use.row].assign(_sid=use.SampleID.to_numpy(), _rank=use.Rank.to_numpy())
    part["_row"] = part.index
    part["_reads"] = pd.to_numeric(part[reads], errors="coerce").fillna(0)
    part = part.sort_values(["_sid", "_rank", "_row"])
    part["_total"] = part.groupby(keys)._reads.transform("sum")
    # Keep each merged sample together, at the place of its first row
    part["_p"] = part.groupby("_sid")._row.transform("min")
    part["_l"] = part.groupby(["_sid", locus])._row.transform("min")
    m = part.drop_duplicates(keys).copy()

    m[reads] = m._total.map(number_text)
    at_locus = m.groupby(["_sid", locus])._total.transform("sum")
    if c["norm"]:
        m[c["norm"]] = (m._total / at_locus.where(at_locus > 0)).map(lambda v: "" if pd.isna(v) else str(v))
    if c["n_alleles"]:
        m[c["n_alleles"]] = m.groupby(["_sid", locus])._total.transform("size").astype(str)
    m[c["id"]] = m._sid.map(part[part._rank == 0].groupby("_sid")[c["id"]].first())
    # Tables (and runs) of the sample, best first
    m[c["table"]] = m._sid.map(part.groupby("_sid", sort=False)[c["table"]].agg(join_unique))
    if c["run"]:
        m[c["run"]] = m._sid.map(part[c["run"]].str.strip().groupby(part._sid, sort=False).agg(join_unique))
    if c["allele"]:
        clash = m.duplicated(["_sid", locus, c["allele"]], keep=False)
        if clash.any():
            print(f"Note: {m[clash].groupby(['_sid', locus, c['allele']]).ngroups} allele names are used for two "
                  "different sequences in the same merged sample, because each table numbers its alleles "
                  "separately. Match alleles by asv, not by allele name.")

    rest = al.drop(index=dup.row)
    rest = rest.assign(_p=rest.index, _l=rest.index, _row=rest.index)
    out = pd.concat([rest, m]).sort_values(["_p", "_l", "_row"], kind="stable")
    return out[list(al.columns)]

## 1. Check the settings

In [ ]:
if isinstance(ALLELE_FILES, str):
    ALLELE_FILES = [ALLELE_FILES]
modes = [name for name, on in [("merge", MERGE), ("best_coverage", BEST_COVERAGE), ("keep_dups", KEEP_DUPS)] if on]
if len(modes) > 1:
    raise ValueError("Set only one of MERGE, BEST_COVERAGE and KEEP_DUPS to True.")
mode = modes[0] if modes else None
if OUT_ALLELES_FILE and not mode:
    raise ValueError("OUT_ALLELES_FILE needs MERGE, BEST_COVERAGE or KEEP_DUPS set to True.")
outputs = [os.path.abspath(x) for x in (OUT_FILE, OUT_ALLELES_FILE) if x]
files = allele_files(ALLELE_FILES)
if {os.path.abspath(f) for f in files} & set(outputs) or len(set(outputs)) < len(outputs):
    raise ValueError("OUT_FILE and OUT_ALLELES_FILE must be different from each other and from the allele tables "
                     "(including the allele* files in a folder in ALLELE_FILES).")

## 2. Read the allele tables

In [ ]:
print(f"Allele tables read ({len(files)}): {shortlist(files)}")
al, table_col = read_alleles(files)
sid = find_col(al, ALLELES_ID_COL, ID_NAMES, "the allele table IDs")
locus = find_col(al, None, ["locus"], "")
reads = find_col(al, None, ["reads"], "")
run = find_col(al, None, ["run"], "")
asv = find_col(al, None, ["asv"], "")
if not (sid and locus and reads):
    raise ValueError(f"The allele tables need sample ID, locus and reads columns. Columns: {list(al.columns)}")
if OUT_ALLELES_FILE and mode == "merge" and not asv:
    raise ValueError("MERGE with OUT_ALLELES_FILE needs an asv column in the allele tables to match alleles "
                     "across tables (allele names are numbered separately in each table).")

# Origin of each row: the allele table it comes from
n_tables = al[table_col].nunique()
data = pd.DataFrame({"row": al.index, "raw": al[sid].str.strip(), "Allele_Table": al[table_col],
                     "run": al[run].str.strip() if run else "", "locus": al[locus],
                     "reads": pd.to_numeric(al[reads], errors="coerce").fillna(0)})
if asv:
    data["asv"] = al[asv]
data["SampleID"] = data.raw.map(normalise_id)
# Only samples are screened; controls (N3D7-10K, NDd2_100k, ...) are left unchanged
is_sample = data.SampleID.str.fullmatch(SAMPLE_ID_PATTERN)
controls = sorted(set(data.SampleID[~is_sample]))
data = data[is_sample]

links = data[["row", "SampleID", "Allele_Table"]]
repeated_rows = set()
if asv:
    # The same sequence listed twice under two allele names would be counted twice
    repeated = data.duplicated(["raw", "Allele_Table", "run", "locus", "asv"])
    if repeated.any():
        print(f"Note: ignored {repeated.sum()} repeated sequence rows (same sample, allele table, locus and asv).")
    repeated_rows = set(data.row[repeated])
    data = data[~repeated]

## 3. Depth of each sample in each allele table

In [ ]:
per_locus = data.groupby(["SampleID", "Allele_Table", "locus"], as_index=False).reads.sum()
stats = summarise(per_locus, ["SampleID", "Allele_Table"], MIN_DEPTH)
raw_ids = data.groupby(["SampleID", "Allele_Table"]).raw.agg(join_sorted).rename("Raw_SampleID")
stats = stats.merge(raw_ids.reset_index(), on=["SampleID", "Allele_Table"])
# Samples sequenced twice inside one table (two sequencing IDs or runs): their reads are added up
inside = data.drop_duplicates(["SampleID", "Allele_Table", "raw", "run"]).groupby(["SampleID", "Allele_Table"]).size()
n_inside = inside[inside > 1].index.get_level_values("SampleID").nunique()
stats["N_Runs"] = stats.groupby("SampleID").Allele_Table.transform("nunique")
stats = stats.sort_values(["SampleID", "Depth_Read_Loci", "Total_Reads", "Allele_Table"],
                          ascending=[True, False, False, True])
stats["Rank"] = stats.groupby("SampleID").cumcount()
stats["Best_Run"] = (stats.Rank == 0).map({True: "yes", False: "no"})
ranks = stats.loc[stats.N_Runs > 1, ["SampleID", "Allele_Table", "Rank"]]
if not SHOW_ALL:
    stats = stats[stats.N_Runs > 1]

## 4. Screening table

One row per sample and allele table, best table first. Only samples in more than one table are listed, unless `SHOW_ALL = True`.

| Column | Meaning |
|---|---|
| `SampleID` | Standardised ID |
| `Allele_Table` | Allele table the sample's rows come from |
| `Raw_SampleID` | ID(s) as written in that table |
| `N_Runs` | Number of allele tables the sample is in |
| `Total_Reads` | Reads over all loci |
| `Total_Loci` | Loci with at least one read |
| `Avg_Reads_per_Locus` | `Total_Reads` / `Total_Loci` |
| `Depth_Read_Loci` | Loci with more than `MIN_DEPTH` reads |
| `Best_Run` | `yes` / `no` (with `MERGE`: the best table) |
| `New_SampleID` | Only with `KEEP_DUPS`: the ID with its letter |

With `BEST_COVERAGE`, only the best row of each sample is kept. With `MERGE`, each sample has one row, and its allele tables and IDs are listed best first, separated by `;`.

In [ ]:
cols = ["SampleID", "Allele_Table", "Raw_SampleID", "N_Runs", "Total_Reads", "Total_Loci",
        "Avg_Reads_per_Locus", "Depth_Read_Loci", "Best_Run"]
if mode == "merge":
    # Add up each sample's reads per locus over its tables, then recount
    pooled = per_locus[per_locus.SampleID.isin(stats.SampleID)]
    pooled = pooled.groupby(["SampleID", "locus"], as_index=False).reads.sum()
    merged = stats.groupby("SampleID", sort=False).agg(
        Allele_Table=("Allele_Table", join_unique), Raw_SampleID=("Raw_SampleID", ";".join),
        N_Runs=("Allele_Table", "size"), Best_Run=("Allele_Table", "first")).reset_index()
    stats = summarise(pooled, ["SampleID"], MIN_DEPTH).merge(merged, on="SampleID").sort_values("SampleID")
elif mode == "best_coverage":
    stats = stats[stats.Rank == 0]
elif mode == "keep_dups":
    stats = stats.assign(New_SampleID=stats.SampleID + ("_" + stats.Rank.map(letter)).where(stats.N_Runs > 1, ""))
    cols.append("New_SampleID")
stats["Total_Reads"] = stats.Total_Reads.round().astype("int64")
# Files are read as latin-1 and written as latin-1, so every byte is kept as it was
stats[cols].to_csv(OUT_FILE, sep=separator(OUT_FILE), index=False, encoding="latin-1")

print(f"Allele tables: {n_tables} | samples: {links.SampleID.nunique()} | "
      f"controls (not screened, left unchanged): {len(controls)}")
if n_tables == 1:
    print("Note: only one allele table, so no sample can be in more than one. Give two or more tables, "
          "or a folder with them.")
if controls:
    print(f"Controls: {shortlist(controls)}")
shape = {"merge": "merged into one row each",
         "best_coverage": "only the best allele table of each kept",
         "keep_dups": "all kept, IDs ending in _a, _b, ... (a = best)"}.get(mode, "listed in the screening table")
print(f"Samples in more than one allele table: {ranks.SampleID.nunique()} ({shape})")
if n_inside:
    print(f"Note: {n_inside} samples are more than once in the same allele table (two sequencing IDs or runs); "
          "their reads are added up as one (see Raw_SampleID).")
print(f"Depth_Read_Loci = loci with more than {MIN_DEPTH:g} reads")
print(f"Written: {OUT_FILE}")

In [ ]:
stats[cols].reset_index(drop=True)

## 5. Allele table

This step runs only when `OUT_ALLELES_FILE` is set. It writes one table with all the input tables combined, with the same columns as the input plus `Allele_Table` (the table each row comes from). Samples in only one allele table, and controls, are unchanged.

| Option | Change in the allele table |
|---|---|
| `BEST_COVERAGE` | Rows from the other tables are removed |
| `KEEP_DUPS` | All rows are kept, and IDs get `_a`, `_b`, ... |
| `MERGE` | One row per sample, locus and `asv`. Reads are summed and `norm.reads.locus` and `n.alleles` recalculated. `Allele_Table` lists all the sample's tables, best first (and `Run`, if the tables have it). |

In [ ]:
if not OUT_ALLELES_FILE:
    if mode:
        print("Note: only the screening table was written; set OUT_ALLELES_FILE to also write the allele table.")
else:
    out = resolve_alleles(al, {"id": sid, "locus": locus, "reads": reads, "asv": asv, "run": run,
                               "nida": find_col(al, None, ["nida"], ""),
                               "allele": find_col(al, None, ["allele"], ""),
                               "norm": find_col(al, None, ["norm.reads.locus"], ""),
                               "n_alleles": find_col(al, None, ["n.alleles"], ""), "table": table_col},
                          links, repeated_rows, ranks, mode)
    out.to_csv(OUT_ALLELES_FILE, sep=separator(OUT_ALLELES_FILE), index=False, encoding="latin-1")
    print(f"Allele table rows: {len(al)} in, {len(out)} out")
    print(f"Written: {OUT_ALLELES_FILE}")